In [1]:
import pandas as pd
import json

In [3]:
SAVE_DIR = "./../data/datasets/"

In [4]:
def unify_dataset_schema(file_json, file_csv, dataset):
    # create df from json
    df1 = pd.read_json(file_json)
    df1 = df1.transpose()
    df2 = pd.read_csv(file_csv)
    # Initialize the new columns
    df1["Indication_approved_extracted"] = None
    df1["Indication_requested_extracted"] = None
    df1["Marketing_authorisation_holder_extracted"] = None
    
    for row in df1.iterrows():
        document_name = row[1].get("Document_name")
        if document_name in df2["Document_name"].values:
            matching_rows = df2[df2["Document_name"] == document_name]
            if not matching_rows.empty:
                matching_row = matching_rows.iloc[0]
                
                df1.loc[row[0], "Indication_approved_extracted"] = matching_row.get("Indication_approved_extracted", None)
                df1.loc[row[0], "Indication_requested_extracted"] = matching_row.get("Indication_requested_extracted", None)
                df1.loc[row[0], "Marketing_authorisation_holder_extracted"] = matching_row.get("Marketing_authorisation_holder_extracted", None)
                df1.loc[row[0], "Dataset"] = dataset
            else:
                # No matching rows found
                df1.loc[row[0], "Indication_approved_extracted"] = None
                df1.loc[row[0], "Indication_requested_extracted"] = None
                df1.loc[row[0], "Marketing_authorisation_holder_extracted"] = None
        else:
            # Document_name not in df2
            df1.loc[row[0], "Indication_approved_extracted"] = None
            df1.loc[row[0], "Indication_requested_extracted"] = None
            df1.loc[row[0], "Marketing_authorisation_holder_extracted"] = None

    df1 = df1.reindex(sorted(df1.columns), axis=1)

    return df1

# EMA

In [5]:
filepath_json = "./../inference/combined/EMA_manually_cleaned.json"
filepath_csv = "./../inference/combined/with_extracted_data/Diseases_manually_cleaned/EMA_manually_cleaned.csv"
merged_EMA = unify_dataset_schema(filepath_json, filepath_csv, dataset="EMA")
merged_EMA.to_csv(SAVE_DIR + "EMA.csv", index=False)
with open(SAVE_DIR + "EMA.json", "w", encoding="utf-8") as out:
    json.dump(merged_EMA.to_dict(orient="index"), out, indent=4, sort_keys=True)

# Swissmedic

In [6]:
filepath_json = "./../inference/combined/SWISSMEDIC_manually_cleaned.json"
filepath_csv = "./../inference/combined/with_extracted_data/Diseases_manually_cleaned/SWISSMEDIC_manually_cleaned.csv"
merged_SWISSMEDIC = unify_dataset_schema(filepath_json, filepath_csv, dataset="SWISSMEDIC")
merged_SWISSMEDIC.to_csv(SAVE_DIR + "SWISSMEDIC.csv", index=False)
with open(SAVE_DIR + "SWISSMEDIC.json", "w", encoding="utf-8") as out:
    json.dump(merged_SWISSMEDIC.to_dict(orient="index"), out, indent=4, sort_keys=True)

# Japan

In [7]:
filepath_json = "./../inference/combined/JAPAN_manually_cleaned.json"
filepath_csv = "./../inference/combined/with_extracted_data/Diseases_manually_cleaned/JAPAN_manually_cleaned.csv"
merged_JAPAN = unify_dataset_schema(filepath_json, filepath_csv, dataset="JAPAN")
merged_JAPAN.to_csv(SAVE_DIR + "JAPAN.csv", index=False)
with open(SAVE_DIR + "JAPAN.json", "w", encoding="utf-8") as out:
    json.dump(merged_JAPAN.to_dict(orient="index"), out, indent=4, sort_keys=True)

# Australia

In [8]:
filepath_json = "./../inference/combined/AUSTRALIA_manually_cleaned.json"
filepath_csv = "./../inference/combined/with_extracted_data/Diseases_manually_cleaned/AUSTRALIA_manually_cleaned.csv"
merged_AUSTRALIA = unify_dataset_schema(filepath_json, filepath_csv, dataset="AUSTRALIA")
merged_AUSTRALIA.to_csv(SAVE_DIR + "AUSTRALIA.csv", index=False)
with open(SAVE_DIR + "AUSTRALIA.json", "w", encoding="utf-8") as out:
    json.dump(merged_AUSTRALIA.to_dict(orient="index"), out, indent=4, sort_keys=True)

# FDA

In [9]:
with open("./../inference/combined/FDA_manually_cleaned.json", "r") as f1:
    data1 = json.load(f1)
with open("./../data/FDA/with_extracted_data_drug_class/FDA.json", "r") as f2:
    data2 = json.load(f2)

lookup = {}
for entry in data2.values():
    ma_number = entry.get("MA_Number")
    drug_class = entry.get("Non_proprietary_name_extracted")
    if ma_number:
        lookup[ma_number] = drug_class

# Enrich data1
for key, entry in data1.items():
    ma_number = entry.get("Marketing_authorisation_number")
    if ma_number and ma_number in lookup:
        entry["Drug_class"] = lookup[ma_number]
        entry["Application_date"] = None
        entry["Application_year"] = None
        entry["Document_name"] = None
        entry["Indication_approved"] = entry.get("Indications_and_usage")
        entry.pop("Indications_and_usage", None) 
        entry["Indication_approved_extracted"] = None
        entry["Indication_requested"] = None
        entry["Indication_requested_extracted"] = None
        entry["Procedure_number"] = None
        entry["Referral_body"] = entry.get("Referral")
        entry.pop("Referral", None)
        entry["Dataset"] = "FDA"
        entry.pop("Origin", None)
    if not ma_number:
        print(f"MA_Number not found for entry: {entry}")
 
# Save to new JSON file
output_path = "./../data/datasets/FDA.json"
with open(output_path, "w", encoding="utf-8") as out:
    json.dump(data1, out, indent=4, sort_keys=True)

print(f"Enriched file saved to {output_path}")

# Save as CSV just in case
df = pd.DataFrame(data1).transpose()
df.to_csv("./../data/datasets/FDA.csv", encoding="utf-8")



Enriched file saved to ./../data/datasets/FDA.json


In [25]:
df.columns

Index(['Marketing_authorisation_number', 'Drug_name', 'Non_proprietary_name',
       'Marketing_authorisation_holder', 'Pharmaceutical_form',
       'Administration_route', 'Decision', 'Decision_date', 'Decision_year',
       'Current_status', 'Nonclinical_abridged', 'Orphan_drug_status',
       'Marketing_authorisation_holder_extracted', 'Drug_class',
       'Disease_class(es)', 'Disease_name(s)', 'Application_date',
       'Application_year', 'Document_name', 'Indication_approved',
       'Indication_approved_extracted', 'Indication_requested',
       'Indication_requested_extracted', 'Procedure_number', 'Referral_body',
       'Dataset'],
      dtype='object')

# HealthCanada

In [14]:
man_cleaned_path = "./../inference/combined/HEALTHCANADA_manually_cleaned.json"
with_add_columns = "./../data/HealthCanada/with_extracted_data_pdfs_disease_name_class/HEALTHCANADA.json"
pdf_dict_path = "./../data/HealthCanada/pdf_dict.json"

with open(man_cleaned_path, "r") as f1:
    data1 = json.load(f1)
with open(with_add_columns, "r") as f2:
    data2 = json.load(f2)
with open(pdf_dict_path, "r") as f3:
    pdf_dict = json.load(f3)

# Create reverse mapping: MA_number -> PDF_digits
ma_to_pdf = {}
for pdf_url, ma_numbers in pdf_dict.items():
    # Extract digits from PDF URL (e.g., "00078801" from "https://pdf.hres.ca/dpd_pm/00078801.PDF")
    pdf_digits = pdf_url.split('/')[-1].replace('.PDF', '')
    for ma_num in ma_numbers:
        ma_to_pdf[ma_num] = pdf_digits

# Enrich data1 using Marketing_authorisation_number -> PDF digits -> extracted data
for key, entry in data1.items():
    ma_number = entry.get("Marketing_authorisation_number")
    if ma_number and ma_number in ma_to_pdf:
        pdf_digits = ma_to_pdf[ma_number]
        if pdf_digits in data2:
            disease_class = data2[pdf_digits].get("Indications_and_usage_disease_class_extracted")
            entry["Disease_class(es)"] = disease_class
            
            indications_approved = data2[pdf_digits].get("Indications_and_usage")
            entry["Indication_approved"] = indications_approved
            
            indication_requested_extracted = data2[pdf_digits].get("Indications_and_usage_disease_name_extracted")
            entry["Indication_requested_extracted"] = indication_requested_extracted
        else:
            entry["Disease_class(es)"] = None
            entry["Indication_approved"] = None
            entry["Indication_requested_extracted"] = None
    else:
        entry["Disease_class(es)"] = None
        entry["Indication_approved"] = None
        entry["Indication_requested_extracted"] = None
    
    entry["Dataset"] = "HEALTHCANADA"

# Add missing columns for consistency with other datasets
for key, entry in data1.items():
    if "Indication_approved_extracted" not in entry:
        entry["Indication_approved_extracted"] = None
    if "Indication_requested" not in entry:
        entry["Indication_requested"] = None

output_path = "./../data/datasets/HEALTHCANADA.json"
with open(output_path, "w", encoding="utf-8") as out:
    json.dump(data1, out, indent=4, sort_keys=True)

# Save as CSV
df = pd.DataFrame(data1).transpose()
df = df.reindex(sorted(df.columns), axis=1)
df.to_csv("./../data/datasets/HEALTHCANADA.csv", index=False, encoding="utf-8")

print(f"HealthCanada dataset saved to {output_path}")



HealthCanada dataset saved to ./../data/datasets/HEALTHCANADA.json
